# 邏輯回歸

你在大學招生處工作，要用申請人的兩次考試分數，判斷錄取機會高不高。第一部用一條直線把錄取和未錄取分開。第二部換到工廠：兩項測試分數決定晶片能不能過，直線分不開，所以把特徵變彎，再用 `lambda` 約束那條邊界不要繞得太厲害。

程式已經寫好，用來讓你看懂模型在做什麼。請 agent 改的格子，標題會寫「實驗」。下面這種格子只有你能寫，agent 不要填、不要改寫：


<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">你的觀察</h3>

</div>


## 給 agent

- 可以改標題含「實驗」的程式格，也可以依學生當下的問題加新的程式格。
- 不要改標題是「你的觀察」的格子，也不要替學生寫結論。
- 不要把 `compute_cost`、`compute_gradient` 重寫成另一套作業，除非學生明確說要改哪一行來做實驗。
- 學生若只說「幫我做完」，先問他現在想驗證門檻還是 `lambda`，再改程式。


## 第一部：兩次考試，能不能錄取

![招生處依兩次考試分數分出錄取與未錄取](images/admission.jpg)

每個申請人有兩次考試分數，以及最後有沒有被錄取。`y = 1` 是錄取，`y = 0` 是未錄取。第一筆兩次分數約 `34.6` 和 `78.0`，結果是未錄取。


這一部最後要做出的，就是下面這條流程：把一位申請人的兩次分數加權相加，經過 sigmoid 變成錄取機會，再跟門檻比。下面一格一格拆開來做。

![一位申請人從兩次分數到預測結果的流程](images/flow.png)

這一部的順序：

- **1.1** 資料：兩群人怎麼分布
- **1.2** sigmoid：把任何數壓到 0 和 1 之間
- **1.3** 單筆的 loss：猜錯要罰多重
- **1.4** Cost：所有人的 loss 平均
- **1.5** 梯度：w、b 該往哪邊調
- **1.6** 梯度下降：一步一步把 Cost 壓低
- **1.7** 決策邊界：機會剛好等於門檻的地方
- **1.8** 實驗：改門檻


### 1.1 資料：兩群人怎麼分布

先跑下一格，看兩群人在圖上怎麼分開。


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

def loadData(filename):
    try:
        here = Path(__vsc_ipynb_file__).resolve().parent
    except NameError:
        here = Path.cwd()
    return np.loadtxt(here / "data" / filename, delimiter=",")

rows = loadData("ex2data1.csv")
X_train = rows[:, :2]
y_train = rows[:, 2]
print("筆數 m =", len(y_train))
print("前 5 筆 [考試1, 考試2, 錄取]", rows[:5])

admitted = y_train == 1
plt.plot(X_train[admitted, 0], X_train[admitted, 1], "k+", label="admitted")
plt.plot(X_train[~admitted, 0], X_train[~admitted, 1], "yo", label="not admitted")
plt.xlabel("Exam 1 score")
plt.ylabel("Exam 2 score")
plt.legend()
plt.show()


<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">你的觀察</h3>
<ol style="color:#1a1a1a;list-style:decimal;padding-left:1.4em;">
<li style="color:#1a1a1a;">兩次考試都比較高的人，看起來比較容易錄取嗎？</li>
<li style="color:#1a1a1a;">有沒有分數不低卻沒被錄取的？</li>
</ol>

</div>


### 1.2 sigmoid：把任何數壓到 0 和 1 之間

直線模型 `z = w·x + b` 算出來可以是任何實數。錄取機會要落在 0 和 1 之間，所以再過一次 sigmoid：

$$
g(z) = \frac{1}{1+e^{-z}}
$$

下一格畫出 `z` 從 -10 到 10 時 `g(z)` 的樣子。`np.clip` 只是避免 `z` 太極端時 `exp` 溢位。

In [ ]:
import math

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

z = np.linspace(-10, 10, 200)
for z_i in [-10, -2, 0, 2, 10]:
    print(f"g({z_i:3d}) = {sigmoid(z_i):.4f}")

plt.plot(z, sigmoid(z))
plt.axhline(0.5, color="gray", linestyle="--")
plt.axvline(0, color="gray", linestyle="--")
plt.xlabel("z")
plt.ylabel("g(z)")
plt.title("sigmoid")
plt.show()

<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">你的觀察</h3>
<ol style="color:#1a1a1a;list-style:decimal;padding-left:1.4em;">
<li style="color:#1a1a1a;"><code style="color:#1a1a1a;">z</code> 很大或很小時，<code style="color:#1a1a1a;">g(z)</code> 會變成多少：</li>
<li style="color:#1a1a1a;"><code style="color:#1a1a1a;">z = 0</code> 時 <code style="color:#1a1a1a;">g(z)</code> 是多少。這時模型覺得這個人會不會錄取：</li>
</ol>

</div>

### 1.3 單筆的 loss：猜錯要罰多重

模型對一個人給出機會 `f = g(w·x + b)`。`f` 接近 1 表示模型覺得很可能錄取，接近 0 表示很可能不錄取。這一筆猜得好不好，用 loss 衡量：

$$
\text{loss}(f, y) = -y\log(f) - (1-y)\log(1-f)
$$

`y = 1` 時只剩 `-log(f)`，`y = 0` 時只剩 `-log(1-f)`。下一格把兩條都畫出來。

In [ ]:
def loss(f, y):
    f = np.clip(f, 1e-15, 1 - 1e-15)
    return -y * np.log(f) - (1 - y) * np.log(1 - f)

f = np.linspace(0.01, 0.99, 200)
plt.plot(f, loss(f, 1), label="y = 1 (admitted)")
plt.plot(f, loss(f, 0), label="y = 0 (not admitted)")
plt.xlabel("f (model's chance of admission)")
plt.ylabel("loss")
plt.legend()
plt.show()

print(f"真的錄取，模型給 0.9：loss = {loss(0.9, 1):.3f}")
print(f"真的錄取，模型給 0.1：loss = {loss(0.1, 1):.3f}")
print(f"沒有錄取，模型給 0.99：loss = {loss(0.99, 0):.3f}")

<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">你的觀察</h3>
<ol style="color:#1a1a1a;list-style:decimal;padding-left:1.4em;">
<li style="color:#1a1a1a;">模型很有把握卻猜錯，和只差一點點猜錯，loss 差多少：</li>
<li style="color:#1a1a1a;">為什麼這種罰法會逼模型不要亂給很有把握的答案：</li>
</ol>

</div>

### 1.4 Cost：所有人的 loss 平均

把每一個人的 loss 加起來取平均，就是這組 `w`、`b` 的 Cost：

$$
J(w,b) = \frac{1}{m}\sum_{i=0}^{m-1} \text{loss}\left(f^{(i)}, y^{(i)}\right)
$$

`w`、`b` 都是 0 時，每個人的機會都是 0.5，Cost 應約 `0.693`。

In [ ]:
def compute_cost(X, y, w, b):
    f = sigmoid(X @ w + b)
    return float(np.mean(loss(f, y)))

zeros_w = np.zeros(X_train.shape[1])
print(f"w=0, b=0 的 Cost = {compute_cost(X_train, y_train, zeros_w, 0.0):.3f}")
print("參考值是 0.693")

test_w = np.array([0.2, 0.2])
print(f"w=[0.2, 0.2], b=-24 的 Cost = {compute_cost(X_train, y_train, test_w, -24.0):.3f}")
print("參考值是 0.218")

### 1.5 梯度：w、b 該往哪邊調

梯度是 Cost 對 `w`、`b` 的斜率。每一項都是「猜的機會減掉答案」再乘上特徵：

$$
\frac{\partial J}{\partial w_j} = \frac{1}{m}\sum_{i=0}^{m-1}\left(f^{(i)} - y^{(i)}\right)x_j^{(i)} \qquad \frac{\partial J}{\partial b} = \frac{1}{m}\sum_{i=0}^{m-1}\left(f^{(i)} - y^{(i)}\right)
$$

`w`、`b` 都是 0 時，`dj_db` 約 `-0.1`。

In [ ]:
def compute_gradient(X, y, w, b):
    m = X.shape[0]
    error = sigmoid(X @ w + b) - y
    dj_dw = (X.T @ error) / m
    dj_db = float(np.sum(error) / m)
    return dj_dw, dj_db

dj_dw, dj_db = compute_gradient(X_train, y_train, zeros_w, 0.0)
print("梯度 dj_dw", dj_dw, "dj_db", round(dj_db, 3))
print("參考值約 dj_dw=[-12.009, -11.263], dj_db=-0.1")

### 1.6 梯度下降：一步一步把 Cost 壓低

訓練就是重複同一件事：算梯度，把 `w`、`b` 往 Cost 下降的方向走一小步。

$$
w_j = w_j - \alpha\frac{\partial J}{\partial w_j} \qquad b = b - \alpha\frac{\partial J}{\partial b}
$$

學習率 `alpha = 0.001`，走 10000 步。這組數字固定，不在這一格改。`b` 從 `-8` 開始，`w` 從接近 0 的隨機數開始（種子固定為 1，所以每次跑出來一樣）。Cost 應從約 `0.96` 降到約 `0.30`。

In [ ]:
def gradient_descent(X, y, w, b, alpha, num_iters, verbose=True):
    w = np.array(w, dtype=float).copy()
    b = float(b)
    history = []
    step = math.ceil(num_iters / 10)
    for i in range(num_iters):
        dj_dw, dj_db = compute_gradient(X, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        history.append(compute_cost(X, y, w, b))
        if verbose and (i % step == 0 or i == num_iters - 1):
            print(f"Iteration {i:4d}: Cost {history[-1]:8.2f}")
    return w, b, history

np.random.seed(1)
initial_w = 0.01 * (np.random.rand(2) - 0.5)
initial_b = -8.0
w, b, history = gradient_descent(
    X_train, y_train, initial_w, initial_b, alpha=0.001, num_iters=10000
)
print("學到的 w, b =", w, b)

plt.plot(history)
plt.xlabel("iteration")
plt.ylabel("Cost")
plt.title("Cost should fall, then flatten")
plt.show()

### 1.7 決策邊界：機會剛好等於門檻的地方

模型給出的是機會，要變成「錄取／不錄取」，還要一個門檻：機會大於等於門檻就預測錄取。門檻 `0.5` 時，訓練準確率約 `92%`。

圖上的分界線是「機會剛好等於門檻」的那些分數，線的一邊模型比較傾向錄取。

In [ ]:
def predict(X, w, b, threshold):
    return (sigmoid(X @ w + b) >= threshold).astype(float)

base_pred = predict(X_train, w, b, 0.5)
print(f"門檻 0.5 的訓練準確率 {np.mean(base_pred == y_train) * 100:.1f}%")
print("參考值約 92%")

plot_x = np.array([X_train[:, 0].min() - 2, X_train[:, 0].max() + 2])
plot_y = -(w[0] * plot_x + b) / w[1]
plt.plot(X_train[admitted, 0], X_train[admitted, 1], "k+", label="admitted")
plt.plot(X_train[~admitted, 0], X_train[~admitted, 1], "yo", label="not admitted")
plt.plot(plot_x, plot_y, c="b", label="threshold 0.5")
plt.xlabel("Exam 1 score")
plt.ylabel("Exam 2 score")
plt.legend()
plt.show()

### 1.8 實驗：改門檻

下一格只改門檻。門檻放低，模型更容易把人預測成錄取；放高則相反。分界線會跟著平移。


In [ ]:
# 實驗：改 threshold，看預測成錄取的人變多還變少
threshold = 0.5

pred = predict(X_train, w, b, threshold)
print("門檻", threshold)
print("預測成錄取", int(pred.sum()), "人，共", len(pred), "人")
print(f"訓練準確率 {np.mean(pred == y_train) * 100:.1f}%")

logit = np.log(threshold / (1 - threshold))
plot_x = np.array([X_train[:, 0].min() - 2, X_train[:, 0].max() + 2])
plot_y = (logit - b - w[0] * plot_x) / w[1]
plt.plot(X_train[admitted, 0], X_train[admitted, 1], "k+", label="admitted")
plt.plot(X_train[~admitted, 0], X_train[~admitted, 1], "yo", label="not admitted")
plt.plot(plot_x, plot_y, c="b", label=f"threshold {threshold}")
plt.xlabel("Exam 1 score")
plt.ylabel("Exam 2 score")
plt.legend()
plt.show()


<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">你的觀察</h3>
<ol style="color:#1a1a1a;list-style:decimal;padding-left:1.4em;">
<li style="color:#1a1a1a;">門檻從 0.5 調高之後，預測成錄取的人變多還變少：</li>
<li style="color:#1a1a1a;">訓練準確率有沒有跟著變好。變好就代表這個門檻比較適合招生嗎：</li>
<li style="color:#1a1a1a;">若你是招生處，你會把門檻放高還是放低：</li>
</ol>

</div>


## 第二部：兩項測試，晶片能不能過

換一批資料。每個晶片有兩項測試分數，`y = 1` 是合格，`y = 0` 是不合格。第一筆約 `0.05` 和 `0.70`，結果是合格。

這一部的順序：

- **2.1** 資料：直線切不開
- **2.2** 特徵映射：讓邊界可以彎
- **2.3** z-score 標準化：讓每一欄差不多大
- **2.4** 正則化的 Cost：w 越大罰越多
- **2.5** 正則化的梯度與梯度下降
- **2.6** 實驗：lambda 的效果


### 2.1 資料：直線切不開

這兩群包在一起，一條直線切不開。先跑下一格看看。


In [ ]:
chip = loadData("ex2data2.csv")
X_chip = chip[:, :2]
y_chip = chip[:, 2]
print("筆數 m =", len(y_chip), "前 5 筆", chip[:5])

passed = y_chip == 1
plt.plot(X_chip[passed, 0], X_chip[passed, 1], "k+", label="pass")
plt.plot(X_chip[~passed, 0], X_chip[~passed, 1], "yo", label="fail")
plt.xlabel("Microchip test 1")
plt.ylabel("Microchip test 2")
plt.legend()
plt.show()


### 2.2 特徵映射：讓邊界可以彎

邏輯回歸的邊界是 `w·x + b = 0`，特徵只有 `x1`、`x2` 時一定是直線。`map_feature` 把兩個分數展開成 6 次以內的所有乘積，例如 `x1²`、`x1·x2`、`x2⁶`，一共 27 個特徵。`b` 仍然單獨放，不放進這 27 個裡。

模型還是邏輯回歸，只是 `w` 變成 27 維；畫回原來的兩軸時，分界就可以是彎的。

In [ ]:
def map_feature(X1, X2, degree=6):
    X1 = np.atleast_1d(X1)
    X2 = np.atleast_1d(X2)
    out = []
    for i in range(1, degree + 1):
        for j in range(i + 1):
            out.append((X1 ** (i - j)) * (X2**j))
    return np.stack(out, axis=1)

def feature_names(degree=6):
    names = []
    for i in range(1, degree + 1):
        for j in range(i + 1):
            parts = [f"x1^{i - j}" if i - j > 1 else "x1" if i - j == 1 else "",
                     f"x2^{j}" if j > 1 else "x2" if j == 1 else ""]
            names.append("·".join(p for p in parts if p))
    return names

X_mapped = map_feature(X_chip[:, 0], X_chip[:, 1])
print("原本的形狀", X_chip.shape, "展開後的形狀", X_mapped.shape, "（應為 118 筆、27 個特徵）")
print("第 1 顆晶片：x1 =", X_chip[0, 0], " x2 =", X_chip[0, 1])
for name, value in list(zip(feature_names(), X_mapped[0]))[:9]:
    print(f"  {name:8s} = {value:9.5f}")
print("  ……後面還有 18 項")

### 2.3 z-score 標準化：讓每一欄差不多大

展開後各欄的大小差很多，次方越高的欄數值越小。大小差太多時，同一個學習率對有些欄太大、對有些欄太小，梯度下降會走得很慢。

z-score 標準化把每一欄減掉自己的平均，再除以自己的標準差，讓每一欄都在 0 附近、標準差都是 1。之後畫邊界時，格點也要用這同一組平均和標準差換算，不能重算。

In [ ]:
def zscore_normalize_features(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    return (X - mu) / sigma, mu, sigma

X_scaled, X_mu, X_sigma = zscore_normalize_features(X_mapped)
print("標準化前，各欄標準差最大", X_mapped.std(axis=0).max().round(4), "最小", X_mapped.std(axis=0).min().round(4))
print("標準化後，各欄標準差最大", X_scaled.std(axis=0).max().round(4), "最小", X_scaled.std(axis=0).min().round(4))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
axes[0].bar(range(27), X_mapped.std(axis=0))
axes[0].set_title("before: std of each feature")
axes[1].bar(range(27), X_scaled.std(axis=0))
axes[1].set_title("after z-score")
for ax in axes:
    ax.set_xlabel("feature index")
plt.tight_layout()
plt.show()

### 2.4 正則化的 Cost：w 越大罰越多

特徵一多，邊界可以彎得很厲害，跟著每一個點繞。正則化在 Cost 後面加一項，只罰 `w`，不罰 `b`：

$$
J(w,b) = \frac{1}{m}\sum_{i=0}^{m-1} \text{loss}\left(f^{(i)}, y^{(i)}\right) + \frac{\lambda}{2m}\sum_{j=0}^{n-1} w_j^2
$$

`lambda` 越大，`w` 越被往 0 拉，邊界越平滑。下一格先用一份 5 筆的小資料確認算得對，再看同一組 `w` 放大時，加上去的那一項怎麼長。

In [ ]:
def compute_cost_reg(X, y, w, b, lambda_):
    m = X.shape[0]
    penalty = (lambda_ / (2 * m)) * float(np.sum(np.square(w)))
    return compute_cost(X, y, w, b) + penalty

np.random.seed(1)
X_tmp = np.random.rand(5, 6)
y_tmp = np.array([0, 1, 0, 1, 0])
w_tmp = np.random.rand(X_tmp.shape[1]) - 0.5
print("正則化後的 cost:", compute_cost_reg(X_tmp, y_tmp, w_tmp, 0.5, 0.7))
print("預期 0.6850849138741673")

for scale in [1, 2, 4]:
    w_big = w_tmp * scale
    plain = compute_cost(X_tmp, y_tmp, w_big, 0.5)
    total = compute_cost_reg(X_tmp, y_tmp, w_big, 0.5, 0.7)
    print(f"w 放大 {scale} 倍：原本的 Cost {plain:.3f}，加上去的罰項 {total - plain:.3f}")

### 2.5 正則化的梯度與梯度下降

Cost 多了一項，梯度也只有 `w` 多一項，`b` 不變：

$$
\frac{\partial J}{\partial w_j} = \frac{1}{m}\sum_{i=0}^{m-1}\left(f^{(i)} - y^{(i)}\right)x_j^{(i)} + \frac{\lambda}{m}w_j \qquad \frac{\partial J}{\partial b} = \frac{1}{m}\sum_{i=0}^{m-1}\left(f^{(i)} - y^{(i)}\right)
$$

梯度下降的步驟和第一部一樣，只是改用這兩個正則化的版本。下一格先對答案，再寫好帶 `lambda` 的梯度下降。

In [ ]:
def compute_gradient_reg(X, y, w, b, lambda_):
    m = X.shape[0]
    dj_dw, dj_db = compute_gradient(X, y, w, b)
    return dj_dw + (lambda_ / m) * w, dj_db

np.random.seed(1)
X_tmp = np.random.rand(5, 3)
y_tmp = np.array([0, 1, 0, 1, 0])
w_tmp = np.random.rand(X_tmp.shape[1])
dj_dw_tmp, dj_db_tmp = compute_gradient_reg(X_tmp, y_tmp, w_tmp, 0.5, 0.7)
print("dj_db:", dj_db_tmp)
print("正則化後的 dj_dw:", dj_dw_tmp.tolist())
print("預期 dj_db: 0.341798994972791")
print("預期 dj_dw: [0.17380012933994293, 0.32007507881566943, 0.10776313396851499]")

def gradient_descent_reg(X, y, w, b, alpha, num_iters, lambda_):
    w = np.array(w, dtype=float).copy()
    b = float(b)
    history = []
    for _ in range(num_iters):
        dj_dw, dj_db = compute_gradient_reg(X, y, w, b, lambda_)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        history.append(compute_cost_reg(X, y, w, b, lambda_))
    return w, b, history

### 2.6 實驗：lambda 的效果

用標準化過的 27 個特徵訓練。`w`、`b` 從 0 開始，學習率 `1`，走 10000 步，比 `lambda = 0`（不正則化）和 `lambda = 1`。想多看一張，可以把 `10` 加進 `lambdas`。

畫邊界時，在兩軸上鋪一片格點，每個格點都經過同樣的 `map_feature` 和標準化，再找出機會剛好是 0.5 的那條線。

In [ ]:
def plot_chip_boundary(ax, w_hat, b_hat, title):
    ax.plot(X_chip[passed, 0], X_chip[passed, 1], "k+")
    ax.plot(X_chip[~passed, 0], X_chip[~passed, 1], "yo")
    u = np.linspace(-1, 1.5, 80)
    v = np.linspace(-1, 1.5, 80)
    z = np.zeros((len(u), len(v)))
    for i, ui in enumerate(u):
        feats = (map_feature(np.full(len(v), ui), v) - X_mu) / X_sigma
        z[i, :] = sigmoid(feats @ w_hat + b_hat)
    ax.contour(u, v, z.T, levels=[0.5], colors="g")
    ax.set_title(title)
    ax.set_xlabel("test 1")
    ax.set_ylabel("test 2")

# 實驗：改 lambdas，看邊界從亂繞變成平滑
lambdas = [0.0, 1.0]
fig, axes = plt.subplots(1, len(lambdas), figsize=(4 * len(lambdas), 4))
for ax, lambda_ in zip(np.atleast_1d(axes), lambdas):
    w_hat, b_hat, history = gradient_descent_reg(
        X_scaled,
        y_chip,
        np.zeros(X_scaled.shape[1]),
        0.0,
        alpha=1.0,
        num_iters=10000,
        lambda_=lambda_,
    )
    pred = predict(X_scaled, w_hat, b_hat, 0.5)
    accuracy = np.mean(pred == y_chip) * 100
    print(
        f"lambda={lambda_}: 最後 Cost {history[-1]:.3f}, "
        f"||w||={np.linalg.norm(w_hat):.1f}, 訓練準確率 {accuracy:.1f}%"
    )
    plot_chip_boundary(ax, w_hat, b_hat, f"lambda={lambda_}")
plt.tight_layout()
plt.show()
print("參考：lambda=0 時 ||w|| 約 13、準確率約 86%；lambda=1 時 ||w|| 約 3.0、準確率約 83%。")

<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">你的觀察</h3>
<ol style="color:#1a1a1a;list-style:decimal;padding-left:1.4em;">
<li style="color:#1a1a1a;"><code style="color:#1a1a1a;">lambda=0</code> 那張圖，邊界有沒有繞著個別的點走。<code style="color:#1a1a1a;">lambda=1</code> 呢：</li>
<li style="color:#1a1a1a;"><code style="color:#1a1a1a;">lambda=0</code> 的訓練準確率比較高，<code style="color:#1a1a1a;">||w||</code> 也比較大。準確率高就代表模型比較好嗎：</li>
<li style="color:#1a1a1a;">把 <code style="color:#1a1a1a;">10</code> 加進 <code style="color:#1a1a1a;">lambdas</code> 再跑，邊界和準確率怎麼變：</li>
<li style="color:#1a1a1a;">邊界若繞著每一個點走，你會擔心下一顆沒見過的晶片被分錯嗎：</li>
</ol>

</div>


實驗本到這裡。接下來用瀏覽器打開同資料夾的 `委託.html`，在 `例行賽贏球.ipynb` 裡做贏球估計。
